<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</center>
</figure>

# **Universidad de Buenos Aires**
## **Facultad de Ciencias Económicas**

### **Taller de Programación para el Análisis de Datos**

#### **Análisis Exploratorio de Datos — E-Commerce**

---

## Contexto

Trabajamos con un dataset de transacciones de una plataforma de e-commerce. Cada fila representa una orden de compra e incluye información sobre el cliente, el producto, los montos y la logística.



## 0 — Carga del dataset

In [ ]:
import pandas as pd
import numpy as np
import kagglehub
import os

# Descarga del dataset desde Kaggle
path = kagglehub.dataset_download("abbas829/ecommerce-sales-dataset")
csv  = [f for f in os.listdir(path) if f.endswith(".csv")][0]

df   = pd.read_csv(os.path.join(path, csv))

print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")

100%|██████████| 108k/108k [00:00<00:00, 20.3MB/s]

Extracting files...
Dataset cargado: 5000 filas x 12 columnas


---

## 1 — Primera mirada al dataset

Antes de calcular cualquier cosa, hay que entender con qué estamos trabajando: cuántas filas, qué columnas, qué tipo de dato tiene cada una.

In [ ]:
# Primeras filas
df.head()

,order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue
0,10001,1/1/2022,1102,Beauty,South,7,373.65,0.28,Wallet,10,4.7,1883.20
1,10002,1/2/2022,1435,Clothing,South,7,47.74,0.09,Card,6,3.9,304.10
2,10003,1/3/2022,1860,Beauty,East,3,311.28,0.31,COD,6,2.5,644.35
3,10004,1/4/2022,1270,Electronics,West,5,524.47,0.02,Wallet,6,1.6,2569.90
4,10005,1/5/2022,1106,Clothing,West,5,139.87,0.33,Wallet,4,4.9,468.56


In [ ]:
df.columns

Index(['order_id', 'order_date', 'customer_id', 'product_category', 'region',
       'quantity', 'unit_price', 'discount', 'payment_method', 'delivery_days',
       'customer_rating', 'revenue'],
      dtype='object')

In [ ]:
# Últimas filas
df.tail()

,order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue
4995,14996,9/5/2035,1289,Clothing,West,2,371.74,0.27,COD,3,2.3,542.74
4996,14997,9/6/2035,1294,Beauty,North,3,553.67,0.14,COD,11,2.2,1428.47
4997,14998,9/7/2035,1450,Beauty,West,3,77.22,0.24,Wallet,6,3.4,176.06
4998,14999,9/8/2035,1903,Electronics,South,4,138.41,0.21,COD,4,2.8,437.38
4999,15000,9/9/2035,1214,Home,East,7,231.58,0.34,COD,5,3.9,1069.90


In [ ]:
# Dimensiones del DataFrame
filas, columnas = df.shape

print(f"Filas   : {filas}")
print(f"Columnas: {columnas}")

Filas   : 5000
Columnas: 12


In [ ]:
# Tipos de dato de cada columna
# object = texto, int64/float64 = numérico
print(df.dtypes)

order_id              int64
order_date           object
customer_id           int64
product_category     object
region               object
quantity              int64
unit_price          float64
discount            float64
payment_method       object
delivery_days         int64
customer_rating     float64
revenue             float64
dtype: object


In [ ]:
df.describe()

,order_id,customer_id,quantity,unit_price,discount,delivery_days,customer_rating,revenue
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,12500.500000,1505.701200,4.044800,308.418774,0.179984,6.118800,2.973980,1021.955148
std,1443.520003,290.836902,2.020398,169.259369,0.101404,3.153264,1.157722,825.584219
min,10001.000000,1000.000000,1.000000,15.150000,0.000000,1.000000,1.000000,11.210000
25%,11250.750000,1253.000000,2.000000,161.895000,0.090000,3.000000,2.000000,354.527500
50%,12500.500000,1510.000000,4.000000,309.890000,0.180000,6.000000,3.000000,796.650000
75%,13750.250000,1761.000000,6.000000,455.557500,0.270000,9.000000,4.000000,1515.690000
max,15000.000000,1999.000000,7.000000,599.960000,0.350000,11.000000,5.000000,4119.330000


---

## 2 — Valores Nan

Los valores `NaN` pueden distorsionar totales, promedios y filtros si no se detectan a tiempo.

In [ ]:
# Cantidad de NaN por columna
nulos = df.isna().sum()
print("Valores faltantes por columna:")
print(nulos)

Valores faltantes por columna:
order_id            0
order_date          0
customer_id         0
product_category    0
region              0
quantity            0
unit_price          0
discount            0
payment_method      0
delivery_days       0
customer_rating     0
revenue             0
dtype: int64


In [ ]:
# Porcentaje de faltantes sobre el total de filas
# Solo mostramos las columnas que realmente tienen faltantes
pct_nulos = (nulos / len(df) * 100).round(2)
if (nulos.sum()>0) :
  print("Porcentaje de faltantes:",pct_nulos,"%" )
  print(pct_nulos[pct_nulos > 0])

---

## 3 — Variables categóricas: frecuencias

Para las columnas categorías, lo primero es ver cuántos valores únicos hay y cuántas veces aparece cada uno.

In [ ]:
print("Frecuencia por categoría de producto:")
print(df["product_category"].value_counts())

Frecuencia por categoría de producto:
product_category
Electronics    1777
Clothing       1531
Home            969
Beauty          723
Name: count, dtype: int64


In [ ]:
print("Porcentaje por categoría:")
print((df["product_category"].value_counts(normalize=True) * 100).round(1))

Porcentaje por categoría:
product_category
Electronics    35.5
Clothing       30.6
Home           19.4
Beauty         14.5
Name: proportion, dtype: float64


In [ ]:
# Estados de las órdenes
print("Regiones de las órdenes:")
print(df["region"].value_counts())

Regiones de las órdenes:
region
West     1289
North    1276
East     1227
South    1208
Name: count, dtype: int64


---

## 4 — Totales y agregaciones

Calculamos métricas globales del negocio: cuánto se vendió, cuántas unidades, cuál fue el ticket promedio.

In [ ]:
# Revenue total del período
revenue_total = df["revenue"].sum()
print(f"Revenue total : ${revenue_total:,.2f}")

Revenue total : $5,109,775.74


In [ ]:
# Ticket promedio por orden
ticket_promedio = df["revenue"].mean()
print(f"Ticket promedio: ${ticket_promedio:,.2f}")

Ticket promedio: $1,021.96


In [ ]:
# Unidades totales vendidas
unidades_total = df["quantity"].sum()
print(f"Unidades vendidas: {unidades_total:,}")

Unidades vendidas: 20,224


In [ ]:
# Revenue total por categoría — groupby + sum + ordenado
revenue_cat = (
    df.groupby("product_category")["revenue"]
    .sum()
    .sort_values(ascending=False)
)
print("Revenue por categoría:")
print(revenue_cat.apply(lambda x: f"${x:,.2f}"))

Revenue por categoría:
product_category
Electronics    $1,829,899.22
Clothing       $1,531,931.72
Home             $982,083.92
Beauty           $765,860.88
Name: revenue, dtype: object


In [ ]:
# Resumen completo por categoría: órdenes, revenue, ticket y unidades
# agg() permite calcular varias métricas en una sola operación
resumen_cat = df.groupby("product_category").agg(
    cant_ordenes  = ("order_id",    "count"),
    revenue_total = ("revenue", "sum"),
    ticket_prom   = ("revenue", "mean"),
    uds_totales   = ("quantity",    "sum")
).round(2).sort_values("revenue_total", ascending=False)

print(resumen_cat)

                  cant_ordenes  revenue_total  ticket_prom  uds_totales
product_category                                                       
Electronics               1777     1829899.22      1029.77         7109
Clothing                  1531     1531931.72      1000.61         6171
Home                       969      982083.92      1013.50         3949
Beauty                     723      765860.88      1059.28         2995


---

## 5 — Filtrado con `loc` e `iloc`

`loc` e `iloc` son los indexadores principales de pandas para seleccionar filas y columnas.

- **`iloc`** selecciona por **posición numérica** (como índices de una lista)
- **`loc`** selecciona por **etiqueta** — nombre del índice o condición booleana

No son métodos (no usan paréntesis): son objetos que interceptan los corchetes `[]`.

In [ ]:
# iloc — selección por posición numérica
# Filas 0 a 4, todas las columnas
df.iloc[0:5]

,order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue
0,10001,1/1/2022,1102,Beauty,South,7,373.65,0.28,Wallet,10,4.7,1883.20
1,10002,1/2/2022,1435,Clothing,South,7,47.74,0.09,Card,6,3.9,304.10
2,10003,1/3/2022,1860,Beauty,East,3,311.28,0.31,COD,6,2.5,644.35
3,10004,1/4/2022,1270,Electronics,West,5,524.47,0.02,Wallet,6,1.6,2569.90
4,10005,1/5/2022,1106,Clothing,West,5,139.87,0.33,Wallet,4,4.9,468.56


In [ ]:
# iloc — fila 10, columnas 0 a 3
# [fila, columna] — ambos por posición
df.iloc[10, 0:4]

,10
order_id,10011
order_date,1/11/2022
customer_id,1466
product_category,Clothing


In [ ]:
# loc — filtrar por condición booleana
# Órdenes de la categoría Electronics
electronics = df.loc[df["product_category"] == "Electronics"]
print(f"Órdenes de Electronics: {len(electronics)}")
electronics.head()

Órdenes de Electronics: 1777


,order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue
3,10004,1/4/2022,1270,Electronics,West,5,524.47,0.02,Wallet,6,1.6,2569.90
5,10006,1/6/2022,1071,Electronics,West,2,264.44,0.20,Card,3,3.3,423.10
11,10012,1/12/2022,1214,Electronics,West,3,384.71,0.14,COD,11,1.9,992.55
17,10018,1/18/2022,1871,Electronics,East,7,399.90,0.29,COD,3,3.8,1987.50
19,10020,1/20/2022,1130,Electronics,East,7,341.46,0.23,Card,11,3.2,1840.47


In [ ]:
# loc — múltiples condiciones
# & = AND,  | = OR
# Importante: cada condición entre paréntesis
filtro = df.loc[
    (df["product_category"] == "Electronics") &
    (df["revenue"] > 500)
]
print(f"Electronics con revenue > 500: {len(filtro)} órdenes")
print(f"Revenue total del segmento   : ${filtro['revenue'].sum():,.2f}")

Electronics con revenue > 500: 1173 órdenes
Revenue total del segmento   : $1,670,330.24


In [ ]:
# loc — filtrar filas Y seleccionar columnas al mismo tiempo
# loc[condición_filas, lista_de_columnas]
canceladas = df.loc[
    df["customer_rating"] == 5,
    ["order_id", "product_category", "revenue", "region"]
]

print(f"Órdenes canceladas : {len(canceladas)}")
print(f"Revenue perdido    : ${canceladas['revenue'].sum():,.2f}")
canceladas.head()

Órdenes canceladas : 73
Revenue perdido    : $67,809.02


,order_id,product_category,revenue,region
177,10178,Clothing,2650.84,East
217,10218,Beauty,1037.40,North
218,10219,Electronics,1325.58,North
265,10266,Clothing,158.97,South
409,10410,Home,1438.97,North


---

## 6 — Análisis por período

Convertimos la columna de fecha a tipo `datetime` para poder extraer año, mes y calcular diferencias entre fechas.

In [ ]:
# Convertir de string a datetime
df["order_date"] = pd.to_datetime(df["order_date"])
df["delivery_days"]  = pd.to_datetime(df["delivery_days"])

# Extraer año y mes de la fecha de orden
df["anio"] = df["order_date"].dt.year
df["mes"]  = df["order_date"].dt.month

print(f"Rango de fechas:")
print(f"  Desde : {df['order_date'].min().date()}")
print(f"  Hasta : {df['order_date'].max().date()}")

Rango de fechas:
  Desde : 2022-01-01
  Hasta : 2035-09-09


In [ ]:
# Revenue total por mes
revenue_mes = (
    df.groupby(["anio", "mes"])["revenue"]
    .sum()
    .round(2)
)
print("Revenue por mes:")
print(revenue_mes)

Revenue por mes:
anio  mes
2022  1      35624.89
      2      29908.89
      3      30309.28
      4      30781.96
      5      41629.24
               ...   
2035  5      33393.87
      6      25205.26
      7      23944.48
      8      29354.74
      9       4952.23
Name: revenue, Length: 165, dtype: float64


---

## 7 — Preguntas concretas de negocio respondidas con pandas



In [ ]:
df.columns

Index(['order_id', 'order_date', 'customer_id', 'product_category', 'region',
       'quantity', 'unit_price', 'discount', 'payment_method', 'delivery_days',
       'customer_rating', 'revenue', 'anio', 'mes'],
      dtype='object')

In [ ]:
# ¿Cuál es la categoría más vendido por unidades?

top_productos = (
    df.groupby("product_category")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
print("Top 5 categorías por unidades vendidas:")
print(top_productos)

Top 5 categorías por unidades vendidas:
product_category
Electronics    7109
Clothing       6171
Home           3949
Beauty         2995
Name: quantity, dtype: int64


In [ ]:
# ¿Cuál es el revenue por región?

revenue_genero = (
    df.groupby("region")["revenue"]
    .agg(["sum", "mean", "count"])
    .rename(columns={"sum": "total", "mean": "promedio", "count": "ordenes"})
    .round(2)
)
print("Revenue por género:")
print(revenue_genero)

Revenue por género:
             total  promedio  ordenes
region                               
East    1236044.23   1007.37     1227
North   1281508.45   1004.32     1276
South   1246640.90   1031.99     1208
West    1345582.16   1043.90     1289


In [ ]:
# ¿Qué método de pago genera más revenue? — top 10
top_paises = (
    df.groupby("payment_method")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print("Top 10 países por revenue:")
print(top_paises.apply(lambda x: f"${x:,.2f}"))

Top 10 países por revenue:
payment_method
Card      $2,366,248.20
COD       $1,788,408.44
Wallet      $955,119.10
Name: revenue, dtype: object


In [ ]:
# Tabla pivot: revenue total por categoría y region
# Combina dos variables categóricas con una métrica numérica

pivot = pd.pivot_table(
    df,
    values  = "revenue",
    index   = "product_category",
    columns   = "region",
    aggfunc = "sum"
).round(2)

print("Revenue por categoría y region:")
print(pivot)

Revenue por categoría y region:
region                 East      North      South       West
product_category                                            
Beauty            174909.11  186213.32  191073.98  213664.47
Clothing          402582.60  364659.43  348363.14  416326.55
Electronics       411803.35  472450.93  487055.68  458589.26
Home              246749.17  258184.77  220148.10  257001.88
